In [7]:
import sys
sys.path.append("../src")

import pandas as pd

In [8]:
movies_features = pd.read_csv("../data/features/movies_content_features.csv")

# Matrix A (Genres)

In [9]:
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.metrics.pairwise import cosine_similarity

In [10]:
vectorizer = CountVectorizer()

genre_matrix = vectorizer.fit_transform(
    movies_features["soup_genres"]
)

similarity_genres = cosine_similarity(
    genre_matrix
)

In [11]:
similarity_genres.shape

(9742, 9742)

# Matrix B (Semantic Embeddings)

In [12]:
from sentence_transformers import SentenceTransformer

In [16]:
model = SentenceTransformer(
    "sentence-transformers/all-MiniLM-L6-v2"
)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

In [17]:
embeddings = model.encode(
    movies_features["soup_tags"].tolist(),
    normalize_embeddings=True,
    show_progress_bar=True
)

Batches:   0%|          | 0/305 [00:00<?, ?it/s]

In [18]:
similarity_tags = cosine_similarity(
    embeddings
)

# Comparison

In [24]:
toy_story_idx = movies_features[
    movies_features["title"] == "Toy Story (1995)"
].index[0]

In [22]:
similarity_genres[toy_story_idx].max(), similarity_tags[toy_story_idx].max()

(np.float64(1.0000000000000002), np.float32(1.0000004))

In [23]:
sorted(
    similarity_tags[toy_story_idx],
    reverse=True
)[1:10]

[np.float32(0.850091),
 np.float32(0.65314794),
 np.float32(0.523324),
 np.float32(0.46094278),
 np.float32(0.45274365),
 np.float32(0.41016063),
 np.float32(0.40879956),
 np.float32(0.4080578),
 np.float32(0.40659818)]

# Matrices combination

In [25]:
similarity_content = (
    0.4 * similarity_genres +
    0.6 * similarity_tags
)

# Manual inspection

In [27]:
scores = list(
    enumerate(similarity_content[toy_story_idx])
)

scores = sorted(
    scores,
    key=lambda x: x[1],
    reverse=True
)

In [ ]:
for idx, score in scores[1:11]:
    print(
        movies_features.iloc[idx]["title"],
        round(score, 3)
    )